# Building Word Embeddings From Scratch

This notebook creates real word vectors for a sample document **without** calling any pretrained embedding model or API. It follows the classic pipeline behind LSA (and a conceptual ancestor of word2vec/GloVe):

1. Write a sample document
2. Tokenize it
3. Build a co-occurrence matrix (how often words appear near each other)
4. Convert counts to PPMI (Positive Pointwise Mutual Information)
5. Reduce with SVD to get a dense vector per word
6. Print the vector for every unique word
7. Write a `cosine_similarity(word1, word2)` function
8. Bonus: nearest-neighbor search and a 2D visualization

In [ ]:
import re
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. The sample document

An original paragraph about RAG. Swap this string out for any text you want — every cell below works unchanged.

In [ ]:
document = """
Retrieval augmented generation combines a retriever and a generator to answer
questions using external documents. The retriever searches a knowledge base
for relevant text chunks. The generator then reads the retrieved chunks and
produces a grounded answer. A good retriever depends on high quality
embeddings. Embeddings map words into vectors so that words with similar
meaning sit close together in vector space. A vector database stores these
embeddings and supports fast similarity search. When a user asks a question
the system embeds the question and searches the vector database for the
closest chunks. The closest chunks become context for the generator. The
generator combines the context with the question and writes an answer. Good
retrieval improves the answer. Bad retrieval produces a wrong answer.
Engineers evaluate retrieval quality using metrics like recall and precision.
Engineers also evaluate the generator using metrics like faithfulness and
relevancy. Building a strong retriever and a strong generator together
produces a strong question answering system.
"""

print(document)

## 2. Tokenize and build the vocabulary

Lowercase everything, strip punctuation, split into words. Every unique token becomes one row in the embedding matrix.

In [ ]:
def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    return tokens

tokens = tokenize(document)
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print(f"Total tokens: {len(tokens)}")
print(f"Unique words (vocab size): {len(vocab)}")
print(vocab)

## 3. Build a co-occurrence matrix

For every word, look at the words within a sliding window on either side and count how often each pair of words shows up near each other. Words that appear in similar contexts end up with similar rows — that's the whole idea behind distributional embeddings ("a word is defined by the company it keeps").

In [ ]:
def build_cooccurrence_matrix(tokens, vocab_size, word2idx, window_size=4):
    matrix = np.zeros((vocab_size, vocab_size), dtype=np.float64)
    n = len(tokens)
    for i, word in enumerate(tokens):
        center_idx = word2idx[word]
        start = max(0, i - window_size)
        end = min(n, i + window_size + 1)
        for j in range(start, end):
            if j == i:
                continue
            context_idx = word2idx[tokens[j]]
            matrix[center_idx, context_idx] += 1.0
    return matrix

cooc_matrix = build_cooccurrence_matrix(tokens, len(vocab), word2idx, window_size=4)
print("Co-occurrence matrix shape:", cooc_matrix.shape)

## 4. Convert counts to PPMI

Raw counts overweight common words like "the". PPMI (Positive Pointwise Mutual Information) reweights each cell by how much more often two words co-occur than you'd expect by chance, and clips negative values to zero.

In [ ]:
def compute_ppmi(cooc_matrix):
    total = cooc_matrix.sum()
    row_sums = cooc_matrix.sum(axis=1, keepdims=True)
    col_sums = cooc_matrix.sum(axis=0, keepdims=True)

    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log((cooc_matrix * total) / (row_sums @ col_sums))
    pmi[np.isnan(pmi)] = 0.0
    pmi[np.isinf(pmi)] = 0.0
    ppmi = np.maximum(pmi, 0.0)
    return ppmi

ppmi_matrix = compute_ppmi(cooc_matrix)

## 5. Reduce with SVD to get dense word vectors

The PPMI matrix is `vocab_size x vocab_size` and sparse. SVD compresses each row down to a small dense vector that keeps the most important structure. Truncated-SVD-on-PPMI is essentially classic LSA, and it's a direct conceptual ancestor of the embedding models (word2vec, GloVe, transformer-based embedding models) used in production RAG pipelines.

In [ ]:
EMBEDDING_DIM = min(20, len(vocab) - 1)

U, S, VT = np.linalg.svd(ppmi_matrix, full_matrices=False)

word_vectors = U[:, :EMBEDDING_DIM] * S[:EMBEDDING_DIM]

print(f"Embedding dimension used: {EMBEDDING_DIM}")
print(f"word_vectors shape: {word_vectors.shape}  (vocab_size x embedding_dim)")

## 6. Print the vector for every unique word

In [ ]:
def get_vector(word):
    if word not in word2idx:
        raise ValueError(f"'{word}' is not in the vocabulary.")
    return word_vectors[word2idx[word]]

for word in vocab:
    vec = get_vector(word)
    print(f"{word:15s} -> {vec}")

## 7. Cosine similarity between two words

In [ ]:
def cosine_similarity(word1, word2):
    v1 = get_vector(word1)
    v2 = get_vector(word2)
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denom == 0:
        return 0.0
    return float(np.dot(v1, v2) / denom)

pairs_to_check = [
    ("retriever", "generator"),
    ("question", "answer"),
    ("embeddings", "vector"),
    ("retrieval", "quality"),
    ("engineers", "metrics"),
    ("retriever", "engineers"),
]

for w1, w2 in pairs_to_check:
    sim = cosine_similarity(w1, w2)
    print(f"cosine_similarity({w1!r}, {w2!r}) = {sim:.3f}")

## Bonus: nearest neighbors for any word

In [ ]:
def most_similar(word, topn=5):
    if word not in word2idx:
        raise ValueError(f"'{word}' is not in the vocabulary.")
    sims = []
    for other in vocab:
        if other == word:
            continue
        sims.append((other, cosine_similarity(word, other)))
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:topn]

for w in ["retriever", "answer", "embeddings"]:
    print(f"\nMost similar to '{w}':")
    for neighbor, score in most_similar(w, topn=5):
        print(f"  {neighbor:15s} {score:.3f}")

## Bonus: 2D visualization of the word vectors

In [ ]:
coords_2d = U[:, :2] * S[:2]

plt.figure(figsize=(11, 8))
plt.scatter(coords_2d[:, 0], coords_2d[:, 1], alpha=0.6)
for i, word in enumerate(vocab):
    plt.annotate(word, (coords_2d[i, 0], coords_2d[i, 1]), fontsize=9)
plt.title("Word embeddings projected to 2D (first 2 SVD components)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.grid(alpha=0.3)
plt.show()

## Where this connects to your RAG project

This from-scratch pipeline (co-occurrence → PPMI → SVD) is the mechanics-level version of what a real embedding model like `sentence-transformers/all-MiniLM-L6-v2` or `text-embedding-3-small` does under the hood — those are trained with neural nets on huge corpora instead of counted from one paragraph, but the end result (a dense vector per word, compared with cosine similarity) is the same idea. Swap `document` for your real corpus and this is exactly what pgvector will do at scale once retrieval is wired up.